# Stage 3 Qwen Model Sweep Clean (Kaggle)

Frozen-model comparison for the clean Stage 3 GT-crop protocol. The prompt, dataset, schema, and evaluator stay fixed; only `model_id` changes.

The notebook runs a one-sample preflight for each candidate first. Full validation runs only if preflight succeeds.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
JSONL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

PROMPT_VERSION = "qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath"
RUN_ID_PREFIX = "stage3_qwen_val_v2_model_sweep_clean"
MAX_PIXELS = 401408

MODEL_RUNS = [
    {
        "name": "qwen25vl_3b_control",
        "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "full_run": True,
        "notes": "clean 3B baseline control",
    },
    {
        "name": "qwen25vl_7b",
        "model_id": "Qwen/Qwen2.5-VL-7B-Instruct",
        "full_run": True,
        "notes": "same-family scale test",
    },
    {
        "name": "qwen25vl_7b_awq",
        "model_id": "Qwen/Qwen2.5-VL-7B-Instruct-AWQ",
        "full_run": True,
        "notes": "quantized 7B fallback if full 7B is memory-limited",
    },
    {
        "name": "qwen3vl_2b",
        "model_id": "Qwen/Qwen3-VL-2B-Instruct",
        "full_run": True,
        "notes": "newer small Qwen3-VL candidate",
    },
    {
        "name": "qwen3vl_4b",
        "model_id": "Qwen/Qwen3-VL-4B-Instruct",
        "full_run": True,
        "notes": "newer mid-small Qwen3-VL candidate",
    },
]

BASELINE_MODEL_NAME = "qwen25vl_3b_control"
MIN_MEANINGFUL_OBJECT_GAIN = 5

print("Model sweep size:", len(MODEL_RUNS))
for spec in MODEL_RUNS:
    print(spec["name"], "->", spec["model_id"])


In [ ]:
def sh(args, cwd: Path | None = None, check: bool = True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run(
        [str(a) for a in args],
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def require_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path

def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


In [ ]:
gpu = sh(["nvidia-smi"], check=False)
if gpu.returncode != 0:
    raise RuntimeError("GPU is required for this model sweep")

py_gpu = sh(["python", "-c", "import torch; print('cuda_available=', torch.cuda.is_available()); print('device_count=', torch.cuda.device_count())"], check=True)


In [ ]:
DATA_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    if (root / JSONL_REL).exists():
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise FileNotFoundError("Could not find clean Stage 3 JSONL in Kaggle inputs")

input_jsonl = require_path(DATA_ROOT / JSONL_REL, "clean val jsonl")
print("DATA_ROOT:", DATA_ROOT)
print("input_jsonl:", input_jsonl)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(["git", "clone", REPO_URL, str(REPO_DIR)])

sh([
    "python", "-m", "pip", "install", "-q", "-U",
    "transformers>=4.57.0", "accelerate", "qwen-vl-utils", "pillow", "pandas", "pyyaml", "tabulate",
], cwd=REPO_DIR)

# AWQ support is attempted, but the sweep is still useful if this optional package is unavailable.
sh(["python", "-m", "pip", "install", "-q", "autoawq"], cwd=REPO_DIR, check=False)

print("Repo ready:", REPO_DIR)


In [ ]:
base_cfg_path = REPO_DIR / "configs/pipeline/stage3_vlm_gt_baseline.yaml"
base_cfg = yaml.safe_load(base_cfg_path.read_text(encoding="utf-8"))

def make_model_config(model_name: str, model_id: str) -> Path:
    cfg = json.loads(json.dumps(base_cfg))
    cfg["input"]["dataset_jsonl"] = str(input_jsonl)
    cfg["prompt"]["version"] = PROMPT_VERSION
    cfg["backend"]["mode"] = "qwen_hf"
    qwen_cfg = cfg["backend"].setdefault("qwen_hf", {})
    qwen_cfg["model_id"] = model_id
    qwen_cfg["max_pixels"] = MAX_PIXELS
    qwen_cfg["torch_dtype"] = "auto"
    qwen_cfg["device_map"] = "auto"
    qwen_cfg["trust_remote_code"] = True
    cfg["run"]["resume"] = False
    cfg_path = REPO_DIR / "configs" / f"stage3_model_sweep_{model_name}.yaml"
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=False), encoding="utf-8")
    return cfg_path

def read_confusion_counts(eval_dir: Path) -> dict:
    path = eval_dir / "confusion_coarse_class.csv"
    if not path.exists():
        return {}
    df = pd.read_csv(path)
    if "gt\\pred" not in df.columns:
        return {}
    out = {}
    for _, row in df.iterrows():
        gt = row["gt\\pred"]
        total = 0
        correct = 0
        for col in df.columns:
            if col == "gt\\pred":
                continue
            value = int(row[col])
            total += value
            if col == gt:
                correct += value
        out[f"{gt}_total"] = total
        out[f"{gt}_correct"] = correct
        out[f"{gt}_recall"] = (correct / total) if total else None
    return out

def metrics_row(model_name: str, model_id: str, run_id: str, preflight_ok: bool, full_ok: bool, error: str | None, notes: str):
    run_dir = REPO_DIR / "outputs/stage3_vlm_baseline_runs" / run_id
    eval_dir = run_dir / "eval"
    row = {
        "model_name": model_name,
        "model_id": model_id,
        "run_id": run_id,
        "preflight_ok": preflight_ok,
        "full_ok": full_ok,
        "error": error or "",
        "notes": notes,
    }
    metrics_path = eval_dir / "metrics.json"
    if metrics_path.exists():
        m = read_json(metrics_path)
        rates = m.get("rates", {})
        f1 = m.get("f1", {})
        counts = m.get("counts", {})
        evaluated = counts.get("evaluated_total")
        coarse_acc = rates.get("coarse_class_accuracy")
        coarse_correct_total = None
        if isinstance(evaluated, int) and isinstance(coarse_acc, (int, float)):
            coarse_correct_total = round(float(coarse_acc) * evaluated)
        row.update({
            "evaluated_total": evaluated,
            "parse_success": rates.get("parse_success_rate"),
            "schema_valid": rates.get("schema_valid_rate"),
            "coarse_acc": coarse_acc,
            "coarse_correct_total": coarse_correct_total,
            "coarse_macro_f1": f1.get("coarse_class_macro_f1"),
            "visibility_acc": rates.get("visibility_accuracy"),
            "visibility_macro_f1": f1.get("visibility_macro_f1"),
            "needs_review_acc": rates.get("needs_review_accuracy"),
            "tag_mean_jaccard": rates.get("tag_mean_jaccard"),
            "pred_ambiguous_rate": rates.get("pred_ambiguous_rate"),
        })
        row.update(read_confusion_counts(eval_dir))
    return row


In [ ]:
rows = []
completed_run_ids = []
for spec in MODEL_RUNS:
    model_name = spec["name"]
    model_id = spec["model_id"]
    cfg_path = make_model_config(model_name, model_id)
    preflight_id = f"{RUN_ID_PREFIX}_{model_name}_preflight"
    full_id = f"{RUN_ID_PREFIX}_{model_name}"
    preflight_ok = False
    full_ok = False
    error = None

    try:
        sh([
            "python", "scripts/run_stage3_vlm_baseline.py",
            "--config", str(cfg_path),
            "--run-id", preflight_id,
            "--max-samples", "1",
            "--no-resume",
        ], cwd=REPO_DIR)
        preflight_ok = True
    except Exception as exc:
        error = f"preflight failed: {exc}"
        print(error)

    if preflight_ok and spec.get("full_run", True):
        try:
            run_dir = REPO_DIR / "outputs/stage3_vlm_baseline_runs" / full_id
            sh([
                "python", "scripts/run_stage3_vlm_baseline.py",
                "--config", str(cfg_path),
                "--run-id", full_id,
                "--no-resume",
            ], cwd=REPO_DIR)
            sh([
                "python", "scripts/validate_vlm_labels_v1.py",
                "--input", str(run_dir / "predictions_vlm_labels_v1.jsonl"),
            ], cwd=REPO_DIR)
            sh([
                "python", "scripts/eval_stage3_vlm_baseline.py",
                "--run-dir", str(run_dir),
                "--ground-truth-jsonl", str(input_jsonl),
                "--output-dir", str(run_dir / "eval"),
            ], cwd=REPO_DIR)
            full_ok = True
            completed_run_ids.append(full_id)
        except Exception as exc:
            error = f"full run failed: {exc}"
            print(error)

    rows.append(metrics_row(model_name, model_id, full_id, preflight_ok, full_ok, error, spec.get("notes", "")))

comparison = pd.DataFrame(rows)
display(comparison)


In [ ]:
def verdict_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    baseline = out[out["model_name"] == BASELINE_MODEL_NAME]
    if baseline.empty or "coarse_correct_total" not in out.columns:
        out["verdict"] = "NO_BASELINE"
        return out
    b = baseline.iloc[0]
    b_correct = b.get("coarse_correct_total")
    b_f1 = b.get("coarse_macro_f1")
    verdicts = []
    for _, row in out.iterrows():
        if not bool(row.get("full_ok")):
            verdicts.append("NOT_RUN")
            continue
        if row.get("model_name") == BASELINE_MODEL_NAME:
            verdicts.append("CONTROL")
            continue
        correct_gain = row.get("coarse_correct_total") - b_correct if pd.notna(row.get("coarse_correct_total")) and pd.notna(b_correct) else None
        f1_gain = row.get("coarse_macro_f1") - b_f1 if pd.notna(row.get("coarse_macro_f1")) and pd.notna(b_f1) else None
        parse_ok = row.get("parse_success") == 1.0 and row.get("schema_valid") == 1.0
        if parse_ok and correct_gain is not None and correct_gain >= MIN_MEANINGFUL_OBJECT_GAIN and f1_gain is not None and f1_gain > 0:
            verdicts.append("STRONG_KEEP")
        elif parse_ok and ((correct_gain is not None and correct_gain > 0) or (f1_gain is not None and f1_gain > 0)):
            verdicts.append("WEAK_SIGNAL")
        else:
            verdicts.append("NO_GAIN")
    out["verdict"] = verdicts
    if "coarse_macro_f1" in out.columns:
        out = out.sort_values(["full_ok", "coarse_macro_f1", "coarse_acc", "tag_mean_jaccard"], ascending=[False, False, False, False])
    return out

comparison = verdict_table(comparison)
display(comparison)


In [ ]:
out_dir = REPO_DIR / "outputs/stage3_model_sweep_clean"
out_dir.mkdir(parents=True, exist_ok=True)
comparison_path = out_dir / "model_sweep_comparison.csv"
comparison.to_csv(comparison_path, index=False)

md_lines = [
    "# Stage 3 Qwen Model Sweep Clean",
    "",
    f"Prompt version: `{PROMPT_VERSION}`",
    f"Max pixels: `{MAX_PIXELS}`",
    "",
    comparison.to_markdown(index=False),
]
(out_dir / "model_sweep_comparison.md").write_text("\n".join(md_lines), encoding="utf-8")

completed = comparison[comparison["full_ok"] == True].copy()
if not completed.empty and "coarse_macro_f1" in completed.columns:
    ranked = completed.sort_values(["coarse_macro_f1", "coarse_acc", "tag_mean_jaccard"], ascending=[False, False, False])
    best = ranked.iloc[0].to_dict()
    print("Best completed model:", best.get("model_name"), best.get("model_id"))
    print("coarse_acc:", best.get("coarse_acc"), "coarse_macro_f1:", best.get("coarse_macro_f1"))
    print("verdict:", best.get("verdict"))
else:
    print("No full model run completed. Check the error column in the comparison table.")

package_root = REPO_DIR / "outputs/stage3_model_sweep_package"
if package_root.exists():
    shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)
shutil.copy2(out_dir / "model_sweep_comparison.csv", package_root / "model_sweep_comparison.csv")
shutil.copy2(out_dir / "model_sweep_comparison.md", package_root / "model_sweep_comparison.md")

runs_out = package_root / "stage3_vlm_baseline_runs"
runs_out.mkdir(parents=True, exist_ok=True)
for run_id in completed_run_ids:
    src = REPO_DIR / "outputs/stage3_vlm_baseline_runs" / run_id
    if src.exists():
        shutil.copytree(src, runs_out / run_id)

archive_path = Path("/kaggle/working/stage3_deliverables_model_sweep_clean.tar.gz")
if archive_path.exists():
    archive_path.unlink()
sh(["tar", "-czf", str(archive_path), "-C", str(package_root.parent), package_root.name], check=True)
print("Archive:", archive_path)
